# CS4.406 A2 — Q3 (baseline/ablation/CI) & Q5 (extended evaluation)

**Input:** `reranked_ebnerd_validation_scores.parquet` and
`reranked_mind_validation_scores.parquet`, handed off from the Q2/Q4 half of
the team. Each row is one impression with the retrieved candidate list,
ground-truth labels, the Stage-1 (retrieval-only) score, and the Stage-2
(re-ranked) score.

**Mapping used for Q3:** "baseline reproduced" = ranking by `stage1_scores`
alone (retrieval score, no click-log-trained signal — the honest analogue of
the starter/official baseline given what data this handoff contains).
"the one principled improvement" = the trained re-ranker's `reranked_scores`
(adds click-log behavioural features on top of retrieval). This is stated
explicitly here so it goes into the design note the same way, rather than
being implied.

**Known gap (flag to your partner, not a blocker for what follows):**
mean candidate-list size is ~12 (EB-NeRD) / ~37 (MIND) — matching your A1
*live-pool* sizes, not a full-catalogue top-150 retrieval. Confirm with him
whether Stage 1 retrieved from the full catalogue or reranked the given pool;
it changes one sentence in the report, nothing in this notebook.

**Known gap (blocks part of Q5 until you get a companion file):**
diversity/novelty/coverage and the cold/warm + head/tail slices need article
category, article popularity, and user history length — none of which are in
this parquet. Section 5 below is written to consume that data the moment you
have it; ask your partner for two small tables:
  - `article_id, category, popularity` (from his article feature store)
  - `user_id, hist_len` (from his `ctx_eval`)


In [2]:
import numpy as np
import polars as pl
from sklearn.metrics import roc_auc_score
from pathlib import Path

def find_one(pattern):
    hits = sorted(Path("/kaggle/input").rglob(pattern))
    if not hits:
        raise FileNotFoundError(f"could not find {pattern} under /kaggle/input — "
                                 f"check the dataset was actually attached to this notebook")
    return hits[0]

DATASETS = {
    "EB-NeRD": find_one("reranked_ebnerd_validation_scores.parquet"),
    "MIND": find_one("reranked_mind_validation_scores.parquet"),
}


dfs = {name: pl.read_parquet(path) for name, path in DATASETS.items()}
for name, df in dfs.items():
    print(name, df.shape, df.columns)


EB-NeRD (5000, 6) ['impression_id', 'user_id', 'candidate_ids', 'labels', 'stage1_scores', 'reranked_scores']
MIND (5000, 6) ['impression_id', 'user_id', 'candidate_ids', 'labels', 'stage1_scores', 'reranked_scores']


## Part I — Q5 core metrics (built first: Q3's ablation is computed from these same per-impression arrays)

In [3]:
# ---------------------------------------------------------------------------
# Per-impression metric functions — standard definitions used by the
# MIND/EB-NeRD official evaluators (binary relevance, AUC/MRR/nDCG).
# ---------------------------------------------------------------------------

def auc_per_impression(labels, scores):
    labels = np.asarray(labels)
    if labels.min() == labels.max():
        return np.nan  # AUC undefined with only one class present
    return roc_auc_score(labels, scores)


def mrr_per_impression(labels, scores):
    order = np.argsort(-np.asarray(scores))
    ranked = np.asarray(labels)[order]
    hits = np.flatnonzero(ranked == 1)
    return np.nan if len(hits) == 0 else 1.0 / (hits[0] + 1)


def _dcg_at_k(labels_sorted, k):
    labels_sorted = np.asarray(labels_sorted[:k], dtype=np.float64)
    discounts = 1.0 / np.log2(np.arange(2, len(labels_sorted) + 2))
    return float(np.sum(labels_sorted * discounts))


def ndcg_at_k(labels, scores, k):
    labels = np.asarray(labels)
    order = np.argsort(-np.asarray(scores))
    dcg = _dcg_at_k(labels[order], k)
    idcg = _dcg_at_k(np.sort(labels)[::-1], k)
    return np.nan if idcg == 0 else dcg / idcg


def per_impression_metrics(df, score_col):
    """Returns a dict of 1D numpy arrays, one value per row/impression,
    aligned to df's row order — this alignment is what makes the PAIRED
    bootstrap in Q3 valid (same impression, same position, both variants).
    """
    aucs, mrrs, ndcg5, ndcg10 = [], [], [], []
    for labels, scores in zip(df["labels"].to_list(), df[score_col].to_list()):
        aucs.append(auc_per_impression(labels, scores))
        mrrs.append(mrr_per_impression(labels, scores))
        ndcg5.append(ndcg_at_k(labels, scores, 5))
        ndcg10.append(ndcg_at_k(labels, scores, 10))
    return dict(auc=np.array(aucs), mrr=np.array(mrrs),
                ndcg5=np.array(ndcg5), ndcg10=np.array(ndcg10))


metrics = {}  # metrics[dataset][stage] -> dict of per-impression arrays
for name, df in dfs.items():
    metrics[name] = {
        "stage1": per_impression_metrics(df, "stage1_scores"),
        "stage2": per_impression_metrics(df, "reranked_scores"),
    }
print("Per-impression metrics computed for both stages, both datasets.")


Per-impression metrics computed for both stages, both datasets.


In [4]:
# ---------------------------------------------------------------------------
# Unpaired bootstrap CI (for the headline "here is our metric with a CI" table)
# ---------------------------------------------------------------------------
def bootstrap_ci(values, n_boot=1000, seed=0):
    values = values[~np.isnan(values)]
    rng = np.random.default_rng(seed)
    n = len(values)
    boot_means = np.array([rng.choice(values, size=n, replace=True).mean()
                            for _ in range(n_boot)])
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    return float(values.mean()), float(lo), float(hi)


rows = []
for name in DATASETS:
    for stage in ("stage1", "stage2"):
        for metric_name, arr in metrics[name][stage].items():
            mean, lo, hi = bootstrap_ci(arr)
            rows.append({"dataset": name, "stage": stage, "metric": metric_name,
                          "mean": round(mean, 4), "ci_lo": round(lo, 4), "ci_hi": round(hi, 4)})

summary_table = pl.DataFrame(rows)
print(summary_table)
summary_table.write_csv("q5_metrics_summary.csv")


shape: (16, 6)
┌─────────┬────────┬────────┬────────┬────────┬────────┐
│ dataset ┆ stage  ┆ metric ┆ mean   ┆ ci_lo  ┆ ci_hi  │
│ ---     ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
│ str     ┆ str    ┆ str    ┆ f64    ┆ f64    ┆ f64    │
╞═════════╪════════╪════════╪════════╪════════╪════════╡
│ EB-NeRD ┆ stage1 ┆ auc    ┆ 0.4846 ┆ 0.4758 ┆ 0.4939 │
│ EB-NeRD ┆ stage1 ┆ mrr    ┆ 0.3069 ┆ 0.2997 ┆ 0.3146 │
│ EB-NeRD ┆ stage1 ┆ ndcg5  ┆ 0.3331 ┆ 0.3241 ┆ 0.3424 │
│ EB-NeRD ┆ stage1 ┆ ndcg10 ┆ 0.4227 ┆ 0.4156 ┆ 0.4302 │
│ EB-NeRD ┆ stage2 ┆ auc    ┆ 0.6957 ┆ 0.6881 ┆ 0.7036 │
│ …       ┆ …      ┆ …      ┆ …      ┆ …      ┆ …      │
│ MIND    ┆ stage1 ┆ ndcg10 ┆ 0.342  ┆ 0.3335 ┆ 0.3516 │
│ MIND    ┆ stage2 ┆ auc    ┆ 0.6067 ┆ 0.5991 ┆ 0.6152 │
│ MIND    ┆ stage2 ┆ mrr    ┆ 0.3264 ┆ 0.3176 ┆ 0.3366 │
│ MIND    ┆ stage2 ┆ ndcg5  ┆ 0.3025 ┆ 0.2935 ┆ 0.3132 │
│ MIND    ┆ stage2 ┆ ndcg10 ┆ 0.3669 ┆ 0.3583 ┆ 0.3767 │
└─────────┴────────┴────────┴────────┴────────┴────────┘


## Part I — Q2's required deliverable, produced here since the inputs already exist: before/after re-ranking table

In [5]:
before_after_rows = []
for name in DATASETS:
    for metric_name in ("auc", "mrr", "ndcg5", "ndcg10"):
        before = np.nanmean(metrics[name]["stage1"][metric_name])
        after = np.nanmean(metrics[name]["stage2"][metric_name])
        before_after_rows.append({
            "dataset": name, "metric": metric_name,
            "before_rerank": round(before, 4), "after_rerank": round(after, 4),
            "delta": round(after - before, 4),
        })

before_after = pl.DataFrame(before_after_rows)
print(before_after)
before_after.write_csv("q2_before_after_rerank.csv")


shape: (8, 5)
┌─────────┬────────┬───────────────┬──────────────┬────────┐
│ dataset ┆ metric ┆ before_rerank ┆ after_rerank ┆ delta  │
│ ---     ┆ ---    ┆ ---           ┆ ---          ┆ ---    │
│ str     ┆ str    ┆ f64           ┆ f64          ┆ f64    │
╞═════════╪════════╪═══════════════╪══════════════╪════════╡
│ EB-NeRD ┆ auc    ┆ 0.4846        ┆ 0.6957       ┆ 0.2111 │
│ EB-NeRD ┆ mrr    ┆ 0.3069        ┆ 0.4578       ┆ 0.1509 │
│ EB-NeRD ┆ ndcg5  ┆ 0.3331        ┆ 0.5172       ┆ 0.1841 │
│ EB-NeRD ┆ ndcg10 ┆ 0.4227        ┆ 0.5683       ┆ 0.1456 │
│ MIND    ┆ auc    ┆ 0.561         ┆ 0.6067       ┆ 0.0457 │
│ MIND    ┆ mrr    ┆ 0.3038        ┆ 0.3264       ┆ 0.0226 │
│ MIND    ┆ ndcg5  ┆ 0.2782        ┆ 0.3025       ┆ 0.0243 │
│ MIND    ┆ ndcg10 ┆ 0.342         ┆ 0.3669       ┆ 0.0249 │
└─────────┴────────┴───────────────┴──────────────┴────────┘


## Part I — Q3. Baseline Reproduced, Then Beaten (paired bootstrap CI)

This is the methodological piece the spec calls out specifically: **"claimed
gains must ship a paired bootstrap 95% CI that excludes zero."** Paired means
the *same* resampled impressions are used for both variants in every
bootstrap draw, so we're testing the CI of the *difference*, not comparing
two separate CIs that happen not to overlap (which is what A1 did — this is
a deliberate upgrade for A2).


In [6]:
def paired_bootstrap_ci_diff(arr_improved, arr_baseline, n_boot=2000, seed=0):
    """Both arrays must be the SAME LENGTH and SAME ROW ORDER (guaranteed
    here since both came from per_impression_metrics on the same dataframe).
    NaNs (undefined AUC on single-class impressions) are dropped pairwise.
    """
    a, b = np.asarray(arr_improved), np.asarray(arr_baseline)
    mask = ~(np.isnan(a) | np.isnan(b))
    a, b = a[mask], b[mask]
    n = len(a)
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)          # SAME resample for both arrays
        diffs[i] = a[idx].mean() - b[idx].mean()
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(diffs.mean()), float(lo), float(hi), n


ablation_rows = []
for name in DATASETS:
    for metric_name in ("auc", "mrr", "ndcg5", "ndcg10"):
        improved = metrics[name]["stage2"][metric_name]
        baseline = metrics[name]["stage1"][metric_name]
        mean_diff, lo, hi, n_used = paired_bootstrap_ci_diff(improved, baseline)
        significant = lo > 0
        ablation_rows.append({
            "dataset": name, "metric": metric_name, "n_impressions": n_used,
            "mean_delta": round(mean_diff, 4), "ci_lo": round(lo, 4), "ci_hi": round(hi, 4),
            "significant (CI excludes 0)": significant,
        })

ablation_table = pl.DataFrame(ablation_rows)
print(ablation_table)
ablation_table.write_csv("q3_ablation_paired_ci.csv")

print()
for r in ablation_rows:
    verdict = "SIGNIFICANT" if r["significant (CI excludes 0)"] else "not significant — CI includes 0"
    print(f"{r['dataset']:8s} {r['metric']:6s}  Δ={r['mean_delta']:+.4f}  "
          f"CI=[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  -> {verdict}")


shape: (8, 7)
┌─────────┬────────┬───────────────┬────────────┬────────┬────────┬─────────────────────────────┐
│ dataset ┆ metric ┆ n_impressions ┆ mean_delta ┆ ci_lo  ┆ ci_hi  ┆ significant (CI excludes 0) │
│ ---     ┆ ---    ┆ ---           ┆ ---        ┆ ---    ┆ ---    ┆ ---                         │
│ str     ┆ str    ┆ i64           ┆ f64        ┆ f64    ┆ f64    ┆ bool                        │
╞═════════╪════════╪═══════════════╪════════════╪════════╪════════╪═════════════════════════════╡
│ EB-NeRD ┆ auc    ┆ 5000          ┆ 0.2111     ┆ 0.1994 ┆ 0.2232 ┆ true                        │
│ EB-NeRD ┆ mrr    ┆ 5000          ┆ 0.1508     ┆ 0.1393 ┆ 0.162  ┆ true                        │
│ EB-NeRD ┆ ndcg5  ┆ 5000          ┆ 0.1839     ┆ 0.1714 ┆ 0.1963 ┆ true                        │
│ EB-NeRD ┆ ndcg10 ┆ 5000          ┆ 0.1456     ┆ 0.1359 ┆ 0.1551 ┆ true                        │
│ MIND    ┆ auc    ┆ 5000          ┆ 0.0457     ┆ 0.0387 ┆ 0.0524 ┆ true                        │
│ MIND

## Part I — Q5. Extended Evaluation — coverage/diversity/novelty and slices

**Blocked on a companion file from your partner** (see the note at the top).
Once you have `article_meta` (columns: `article_id`, `category`,
`popularity`) and `user_hist_len` (columns: `user_id`, `hist_len`), run the
cell below unmodified — it's written against exactly that shape.


In [7]:
# ---------------------------------------------------------------------------
# Fill these in once your partner sends the companion tables. Until then this
# cell will raise a clear FileNotFoundError rather than silently skip.
# ---------------------------------------------------------------------------
try:
    article_meta = pl.read_parquet("article_meta.parquet")      # article_id, category, popularity
    user_hist_len = pl.read_parquet("user_hist_len.parquet")    # user_id, hist_len
    HAVE_METADATA = True
except FileNotFoundError:
    print("article_meta.parquet / user_hist_len.parquet not found yet — "
          "ask your partner for these two small tables, then re-run this cell.")
    HAVE_METADATA = False


article_meta.parquet / user_hist_len.parquet not found yet — ask your partner for these two small tables, then re-run this cell.


In [8]:
# ---------------------------------------------------------------------------
# Coverage / novelty (need popularity); diversity (needs category, as a
# cheap proxy for topical similarity — swap in embedding cosine if your
# partner can also export the 64-d LSA/EMB vectors per article_id).
# ---------------------------------------------------------------------------
def top5_beyond_accuracy(df, score_col, article_meta, n_catalogue):
    pop_map = dict(zip(article_meta["article_id"], article_meta["popularity"]))
    cat_map = dict(zip(article_meta["article_id"], article_meta["category"]))

    seen_articles = set()
    novelty_scores, diversity_scores = [], []

    for cand_ids, scores in zip(df["candidate_ids"].to_list(), df[score_col].to_list()):
        order = np.argsort(-np.asarray(scores))[:5]
        top5 = [cand_ids[i] for i in order]
        seen_articles.update(top5)

        pops = [pop_map.get(a, 0.0) for a in top5]
        # novelty: inverse popularity, higher = more novel (standard beyond-accuracy definition)
        novelty_scores.append(np.mean([-np.log2(p + 1e-9) for p in pops]))

        cats = [cat_map.get(a) for a in top5]
        # intra-list diversity: fraction of pairs with DIFFERENT category
        pairs = [(cats[i] != cats[j]) for i in range(5) for j in range(i + 1, 5)]
        diversity_scores.append(np.mean(pairs) if pairs else np.nan)

    coverage = len(seen_articles) / n_catalogue
    return dict(novelty=float(np.nanmean(novelty_scores)),
                diversity=float(np.nanmean(diversity_scores)),
                coverage=coverage)


if HAVE_METADATA:
    N_CATALOGUE = article_meta["article_id"].n_unique()
    beyond_rows = []
    for name in DATASETS:
        for stage, col in [("stage1", "stage1_scores"), ("stage2", "reranked_scores")]:
            r = top5_beyond_accuracy(dfs[name], col, article_meta, N_CATALOGUE)
            r.update(dataset=name, stage=stage)
            beyond_rows.append(r)
    beyond_table = pl.DataFrame(beyond_rows)
    print(beyond_table)
    beyond_table.write_csv("q5_beyond_accuracy.csv")


In [9]:
# ---------------------------------------------------------------------------
# Slices: cold-start vs warm (needs hist_len), head vs tail (needs popularity
# of the CLICKED article specifically).
# ---------------------------------------------------------------------------
def sliced_auc(df, score_col, hist_map, pop_map, cold_thresh=5, head_quantile=0.8):
    aucs_cold, aucs_warm, aucs_head, aucs_tail = [], [], [], []

    pops_all = np.array(list(pop_map.values()))
    head_thr = np.quantile(pops_all, head_quantile) if len(pops_all) else 0.0

    for row in df.iter_rows(named=True):
        labels, scores = row["labels"], row[score_col]
        auc = auc_per_impression(labels, scores)
        hlen = hist_map.get(row["user_id"], 0)
        (aucs_cold if hlen < cold_thresh else aucs_warm).append(auc)

        clicked_idx = [i for i, l in enumerate(labels) if l == 1]
        if clicked_idx:
            clicked_pop = np.mean([pop_map.get(row["candidate_ids"][i], 0.0) for i in clicked_idx])
            (aucs_head if clicked_pop >= head_thr else aucs_tail).append(auc)

    def _summ(a):
        a = np.array([x for x in a if not np.isnan(x)])
        return (np.nan, np.nan) if len(a) == 0 else (a.mean(), len(a))

    return dict(cold=_summ(aucs_cold), warm=_summ(aucs_warm),
                head=_summ(aucs_head), tail=_summ(aucs_tail))


if HAVE_METADATA:
    hist_map = dict(zip(user_hist_len["user_id"], user_hist_len["hist_len"]))
    pop_map_full = dict(zip(article_meta["article_id"], article_meta["popularity"]))

    slice_rows = []
    for name in DATASETS:
        for stage, col in [("stage1", "stage1_scores"), ("stage2", "reranked_scores")]:
            s = sliced_auc(dfs[name], col, hist_map, pop_map_full)
            for slice_name, (auc, n) in s.items():
                slice_rows.append({"dataset": name, "stage": stage, "slice": slice_name,
                                     "auc": round(auc, 4) if not np.isnan(auc) else None, "n": n})
    slice_table = pl.DataFrame(slice_rows)
    print(slice_table)
    slice_table.write_csv("q5_slices.csv")


## Summary of what's committed by this notebook

| File written | Answers |
|---|---|
| `q5_metrics_summary.csv` | Q5 — AUC/MRR/nDCG@5/10, both stages, both datasets, with CIs |
| `q2_before_after_rerank.csv` | Q2.4 — "report metrics before and after re-ranking" |
| `q3_ablation_paired_ci.csv` | Q3 — the paired-bootstrap significance test the spec requires |
| `q5_beyond_accuracy.csv` | Q5 — diversity/novelty/coverage (once companion file arrives) |
| `q5_slices.csv` | Q5 — cold/warm + head/tail slices (once companion file arrives) |

**Still outside this notebook's scope, and still your partner's side:**
Codabench submission of the actual test-set predictions (this parquet is the
*validation* set only) — that inference + submission pipeline stays with
whoever built the Stage-1/Stage-2 scoring functions, since it needs the live
test-set candidate generation, not just these saved validation scores.
